# RAG Pipeline Debug Notebook

Inspect every step of the retrieval pipeline for a given query:
1. **BM25 scores** — tokenization, synonym expansion, raw scores
2. **Semantic scores** — FAISS embedding similarity
3. **RRF fusion** — combined ranking with heading/label/URL boosts
4. **Cross-encoder reranking** — final precision scoring
5. **Page chunk replacement** — when 3+ hits from same page trigger injection

In [61]:
import sys, os, json, re
import numpy as np
from collections import Counter

# Set working directory to backend/
BACKEND_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__ if '__file__' in dir() else '.')), '..')
# Fallback: use notebook path to find backend/
if not os.path.exists(os.path.join(BACKEND_DIR, "data")):
    BACKEND_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(BACKEND_DIR, "data")):
    # Hard path fallback
    BACKEND_DIR = "/Users/pengxiao/Downloads/UChicago_ADS_RAG/backend"

os.chdir(BACKEND_DIR)
sys.path.insert(0, BACKEND_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Data dir exists: {os.path.exists('data/chunked_documents_dedup.json')}")

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

from retrieval import (
    tokenize_for_bm25, expand_query, is_list_query, _requested_count,
    _detect_query_labels, _get_url, _is_dup
)

# Load data
print("Loading chunks...")
with open("data/chunked_documents_dedup.json") as f:
    chunk_records = json.load(f)
print(f"  {len(chunk_records)} chunks loaded")

print("Loading embedder...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Loading FAISS index...")
faiss_index = faiss.read_index("data/uchicago_ads_faiss_dedup.index")

print("Building BM25 index...")
tokenized = [tokenize_for_bm25(c["text"]) for c in chunk_records]
bm25 = BM25Okapi(tokenized)

print("Loading cross-encoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ All resources loaded.")

Working directory: /Users/pengxiao/Downloads/UChicago_ADS_RAG/backend
Data dir exists: True
Loading chunks...
  1012 chunks loaded
Loading embedder...
Loading FAISS index...
Building BM25 index...
Loading cross-encoder...
✅ All resources loaded.


## 🔧 Set your query here

In [76]:
# ✏️ Change this to debug any query
QUERY = "What language will I learn at this program?"

# Pipeline parameters
RETRIEVE_K = 15
RERANK_K = 5
PAGE_CHUNK_THRESHOLD = 3

## Step 1: Query Analysis

Tokenization, synonym expansion, list detection, label intent.

In [77]:
tokens = tokenize_for_bm25(QUERY)
expanded = expand_query(QUERY)
expanded_tokens = tokenize_for_bm25(expanded)
listing = is_list_query(QUERY)
req_count = _requested_count(QUERY)
query_labels = _detect_query_labels(QUERY)

print(f"Query:            {QUERY}")
print(f"BM25 tokens:      {tokens}")
print(f"Expanded query:   {expanded}")
print(f"Expanded tokens:  {expanded_tokens}")
print(f"Is list query:    {listing}")
print(f"Requested count:  {req_count}")
print(f"Detected labels:  {query_labels}")

Query:            What language will I learn at this program?
BM25 tokens:      ['what', 'language', 'learn', 'program']
Expanded query:   What language will I learn at this program?
Expanded tokens:  ['what', 'language', 'learn', 'program']
Is list query:    False
Requested count:  0
Detected labels:  set()


## Step 2: BM25 Scores (Top 20)

In [78]:
bm25_scores = bm25.get_scores(tokenize_for_bm25(expanded))
bm25_top = np.argsort(-bm25_scores)[:20]

def short_url(url):
    """Shorten URL: keep only the last path segment."""
    return url.rstrip("/").split("/")[-1] if url else ""

print(f"{'Rank':<5} {'Score':<8} {'Type':<18} {'URL':<30} {'Chunk ID'}")
print("-" * 120)
for rank, idx in enumerate(bm25_top):
    c = chunk_records[idx]
    ctype = c["metadata"].get("chunk_type", "")
    url = short_url(_get_url(c["metadata"]))[:29]
    print(f"{rank+1:<5} {bm25_scores[idx]:<8.3f} {ctype:<18} {url:<30} {c['chunk_id']}")

Rank  Score    Type               URL                            Chunk ID
------------------------------------------------------------------------------------------------------------------------
1     8.053    section            faqs                           chunk_787_FAQs_|_DSI
2     7.571    section            applying-to-the-ms-in-applied  chunk_1143_Applying_to_the_MS_in_Applied_Data_Science_Program?_Here’s_What_We_Look_For_|_DSI
3     7.240    page               could-ai-help-us-be-more-thou  chunk_386_Could_AI_Help_Us_Be_More_Thoughtful_Voters?_|_DSI
4     6.921    section            mscapp-conference-invites-con  chunk_884_MSCAPP_Conference_Invites_Conversation_About_AI_Research,_Ethics,_and_Equality_|_DSI
5     6.698    section            could-ai-help-us-be-more-thou  chunk_389_Could_AI_Help_Us_Be_More_Thoughtful_Voters?_|_DSI
6     6.086    accordion_course   in-person-program              chunk_473_In-Person_Program_|_DSI
7     5.690    section            could-ai-help-us-b

## Step 3: Semantic Scores (Top 20)

In [79]:
q_emb = embedder.encode([QUERY], normalize_embeddings=True)[0].astype("float32")
all_idxs = list(range(len(chunk_records)))
sub_vecs = np.stack([faiss_index.reconstruct(i) for i in all_idxs])
sem_scores = sub_vecs @ q_emb

sem_top = np.argsort(-sem_scores)[:20]

print(f"{'Rank':<5} {'Score':<8} {'Type':<18} {'URL':<30} {'Chunk ID'}")
print("-" * 120)
for rank, idx in enumerate(sem_top):
    c = chunk_records[idx]
    ctype = c["metadata"].get("chunk_type", "")
    url = short_url(_get_url(c["metadata"]))[:29]
    print(f"{rank+1:<5} {sem_scores[idx]:<8.4f} {ctype:<18} {url:<30} {c['chunk_id']}")

Rank  Score    Type               URL                            Chunk ID
------------------------------------------------------------------------------------------------------------------------
1     0.4007   section            applying-to-the-ms-in-applied  chunk_1141_Applying_to_the_MS_in_Applied_Data_Science_Program?_Here’s_What_We_Look_For_|_DSI
2     0.3816   accordion          course-progressions            chunk_644_Course_Progressions_|_DSI
3     0.3813   accordion          course-progressions            chunk_636_Course_Progressions_|_DSI
4     0.3790   accordion          course-progressions            chunk_630_Course_Progressions_|_DSI
5     0.3771   accordion_course   in-person-program              chunk_449_In-Person_Program_|_DSI
6     0.3745   section            applying-to-the-ms-in-applied  chunk_1142_Applying_to_the_MS_in_Applied_Data_Science_Program?_Here’s_What_We_Look_For_|_DSI
7     0.3684   section            data4all                       chunk_302_Data4All_Hig

## Step 4: RRF Fusion + Boosts

Shows the raw RRF score, then the effect of each boost (heading, label, URL penalty).

In [80]:
# --- RRF fusion ---
k = 60
sem_ranks = np.argsort(-sem_scores)
bm25_ranks = np.argsort(-bm25_scores)
rrf = np.zeros(len(all_idxs))
for rank, rel in enumerate(sem_ranks):
    rrf[rel] += 1.0 / (k + rank + 1)
for rank, rel in enumerate(bm25_ranks):
    rrf[rel] += 1.0 / (k + rank + 1)

rrf_raw = rrf.copy()

# --- Heading boost ---
q_tokens = set(re.sub(r"[^a-z0-9\s]", " ", QUERY.lower()).split())
q_tokens.update(expanded.lower().split())
heading_boosts = np.zeros(len(all_idxs))
for idx in all_idxs:
    heading = (chunk_records[idx]["metadata"].get("heading") or "").lower()
    if not heading:
        continue
    h_tokens = set(heading.split())
    overlap = len(q_tokens & h_tokens) / max(1, len(h_tokens))
    if overlap >= 0.5:
        ctype = chunk_records[idx]["metadata"].get("chunk_type", "")
        multiplier = 2.0 if ctype in ("accordion", "accordion_faq", "page") else 1.0
        boost = overlap * 0.08 * multiplier
        heading_boosts[idx] = boost
        rrf[idx] += boost

# --- Label boost ---
label_boosts = np.zeros(len(all_idxs))
if query_labels:
    for idx in all_idxs:
        chunk_labels = set(chunk_records[idx]["metadata"].get("labels", []))
        if chunk_labels & query_labels:
            label_boosts[idx] = 0.03
            rrf[idx] += 0.03

rrf_before_url = rrf.copy()

# --- Soft URL penalty ---
url_penalties = np.ones(len(all_idxs))
url_seen = {}
for rel in np.argsort(-rrf):
    meta = chunk_records[rel]["metadata"]
    url = _get_url(meta)
    url_seen.setdefault(url, 0)
    if url_seen[url] > 0:
        penalty = 0.9 ** url_seen[url]
        url_penalties[rel] = penalty
        rrf[rel] *= penalty
    url_seen[url] += 1

# --- Show top 30 with breakdown ---
rrf_top = np.argsort(-rrf)[:30]

print(f"{'Rank':<5} {'RRF_raw':<9} {'+Head':<7} {'+Label':<8} {'URLpen':<8} {'Final':<9} {'Type':<15} {'URL':<30} {'Chunk ID'}")
print("-" * 150)
for rank, idx in enumerate(rrf_top):
    c = chunk_records[idx]
    ctype = c["metadata"].get("chunk_type", "")[:14]
    url = short_url(_get_url(c["metadata"]))[:29]
    print(f"{rank+1:<5} {rrf_raw[idx]:<9.5f} {heading_boosts[idx]:<7.4f} {label_boosts[idx]:<8.4f} {url_penalties[idx]:<8.3f} {rrf[idx]:<9.5f} {ctype:<15} {url:<30} {c['chunk_id']}")

Rank  RRF_raw   +Head   +Label   URLpen   Final     Type            URL                            Chunk ID
------------------------------------------------------------------------------------------------------------------------------------------------------
1     0.01054   0.0800  0.0000   1.000    0.09054   accordion_faq   faqs                           chunk_771_FAQs_|_DSI
2     0.00632   0.0800  0.0000   0.900    0.07769   accordion       faqs                           chunk_766_FAQs_|_DSI
3     0.00544   0.0800  0.0000   0.810    0.06921   accordion_faq   faqs                           chunk_767_FAQs_|_DSI
4     0.00538   0.0800  0.0000   0.729    0.06224   accordion_faq   faqs                           chunk_770_FAQs_|_DSI
5     0.01986   0.0400  0.0000   1.000    0.05986   section         in-person-program              chunk_484_In-Person_Program_|_DSI
6     0.01665   0.0400  0.0000   1.000    0.05665   section         %20                            chunk_525_Online_Program_|_DS

## Step 5: Dedup + Top-K Selection

After RRF, near-duplicate chunks are removed and top-K candidates are selected for reranking.

In [81]:
order = np.argsort(-rrf)
candidates = []
seen_texts = []

for rel in order:
    if len(candidates) >= RETRIEVE_K:
        break
    text = chunk_records[rel]["text"]
    if any(_is_dup(text, s, 0.95) for s in seen_texts):
        continue
    seen_texts.append(text)
    meta = chunk_records[rel]["metadata"]
    candidates.append({
        "chunk_id": chunk_records[rel]["chunk_id"],
        "text": text,
        "url": _get_url(meta),
        "page_title": meta.get("page_title", ""),
        "labels": meta.get("labels", []),
        "score": float(rrf[rel]),
        "_idx": int(rel),
    })

print(f"Top {RETRIEVE_K} candidates for reranking (after dedup):\n")
print(f"{'#':<4} {'RRF':<9} {'Type':<15} {'Page Title':<40} {'Chunk ID'}")
print("-" * 120)
for i, h in enumerate(candidates):
    c = chunk_records[h["_idx"]]
    ctype = c["metadata"].get("chunk_type", "")[:14]
    pt = h["page_title"][:39]
    print(f"{i+1:<4} {h['score']:<9.5f} {ctype:<15} {pt:<40} {h['chunk_id'][:55]}")

Top 15 candidates for reranking (after dedup):

#    RRF       Type            Page Title                               Chunk ID
------------------------------------------------------------------------------------------------------------------------
1    0.09054   accordion_faq   FAQs | DSI                               chunk_771_FAQs_|_DSI
2    0.07769   accordion       FAQs | DSI                               chunk_766_FAQs_|_DSI
3    0.06921   accordion_faq   FAQs | DSI                               chunk_767_FAQs_|_DSI
4    0.06224   accordion_faq   FAQs | DSI                               chunk_770_FAQs_|_DSI
5    0.05986   section         In-Person Program | DSI                  chunk_484_In-Person_Program_|_DSI
6    0.05665   section         Online Program | DSI                     chunk_525_Online_Program_|_DSI
7    0.05591   accordion_faq   FAQs | DSI                               chunk_769_FAQs_|_DSI
8    0.05310   section         In-Person Program | DSI                  chun

## Step 6: Cross-Encoder Reranking

Re-scores each (query, chunk) pair with cross-encoder. This is the most important precision step.

In [82]:
pairs = [(QUERY, h["text"]) for h in candidates]
ce_scores = cross_encoder.predict(pairs)

# Show all candidates with CE scores, sorted by CE score
ranked = sorted(zip(ce_scores, candidates), key=lambda x: -x[0])

print(f"{'Rank':<5} {'CE Score':<10} {'RRF':<9} {'Page Title':<40} {'Chunk ID'}")
print("-" * 120)
for i, (score, h) in enumerate(ranked):
    marker = " ✅" if i < RERANK_K else ""
    pt = h["page_title"][:39]
    print(f"{i+1:<5} {score:<10.4f} {h['score']:<9.5f} {pt:<40} {h['chunk_id'][:50]}{marker}")

print(f"\n--- Top {RERANK_K} selected (✅) ---")
hits = [h for _, h in ranked[:RERANK_K]]

Rank  CE Score   RRF       Page Title                               Chunk ID
------------------------------------------------------------------------------------------------------------------------
1     -4.3769    0.07769   FAQs | DSI                               chunk_766_FAQs_|_DSI ✅
2     -4.7968    0.04977   Online Program | DSI                     chunk_531_Online_Program_|_DSI ✅
3     -4.8767    0.05986   In-Person Program | DSI                  chunk_484_In-Person_Program_|_DSI ✅
4     -5.1916    0.04053   In-Person Program | DSI                  chunk_478_In-Person_Program_|_DSI ✅
5     -5.9130    0.06921   FAQs | DSI                               chunk_767_FAQs_|_DSI ✅
6     -5.9461    0.05310   In-Person Program | DSI                  chunk_483_In-Person_Program_|_DSI
7     -6.0709    0.05591   FAQs | DSI                               chunk_769_FAQs_|_DSI
8     -6.3264    0.04041   FAQs | DSI                               chunk_773_FAQs_|_DSI
9     -6.4357    0.06224   FAQs

## Step 7: Page Chunk Replacement

If 3+ hits come from the same page, replace them with the page-level chunk.

In [83]:
page_counts = Counter(h["page_title"] for h in hits if h.get("page_title"))

print("Page title counts in top-K hits:")
for pt, count in page_counts.most_common():
    trigger = " ← REPLACE" if count >= PAGE_CHUNK_THRESHOLD else ""
    print(f"  {count}x  {pt}{trigger}")

# Build page chunk lookup
page_chunks = {}
for c in chunk_records:
    if c["metadata"].get("chunk_type") == "page":
        pt = c["metadata"].get("page_title", "")
        if pt:
            page_chunks[pt] = c

replaced_pages = set()
for pt, count in page_counts.items():
    if count >= PAGE_CHUNK_THRESHOLD and pt in page_chunks:
        replaced_pages.add(pt)

if replaced_pages:
    result = []
    for pt in replaced_pages:
        pc = page_chunks[pt]
        meta = pc["metadata"]
        url = meta.get("source_urls", [meta.get("source_url", "")])[0] if "source_urls" in meta else meta.get("source_url", "")
        result.append({
            "chunk_id": pc["chunk_id"],
            "text": pc["text"],
            "url": url,
            "page_title": meta.get("page_title", ""),
            "labels": meta.get("labels", []),
            "score": 1.0,
        })
    for h in hits:
        if h.get("page_title") not in replaced_pages:
            result.append(h)
    final_hits = result
    print(f"\n🔄 Replaced {sum(page_counts[pt] for pt in replaced_pages)} section chunks → {len(replaced_pages)} page chunk(s)")
else:
    final_hits = hits
    print("\nNo replacement triggered (no page with 3+ hits)")

print(f"\n{'#':<4} {'Type':<15} {'Page Title':<45} {'Chunk ID'}")
print("-" * 110)
for i, h in enumerate(final_hits):
    # find chunk type
    ctype = "page (injected)" if h.get("score") == 1.0 and any(h["chunk_id"] == page_chunks.get(h["page_title"], {}).get("chunk_id") for _ in [1]) else "section"
    for c in chunk_records:
        if c["chunk_id"] == h["chunk_id"]:
            ctype = c["metadata"].get("chunk_type", "")
            break
    pt = h["page_title"][:44]
    print(f"{i+1:<4} {ctype:<15} {pt:<45} {h['chunk_id'][:55]}")

Page title counts in top-K hits:
  2x  FAQs | DSI
  2x  In-Person Program | DSI
  1x  Online Program | DSI

No replacement triggered (no page with 3+ hits)

#    Type            Page Title                                    Chunk ID
--------------------------------------------------------------------------------------------------------------
1    accordion       FAQs | DSI                                    chunk_766_FAQs_|_DSI
2    section         Online Program | DSI                          chunk_531_Online_Program_|_DSI
3    section         In-Person Program | DSI                       chunk_484_In-Person_Program_|_DSI
4    section         In-Person Program | DSI                       chunk_478_In-Person_Program_|_DSI
5    accordion_faq   FAQs | DSI                                    chunk_767_FAQs_|_DSI


## Step 8: Final Chunks Sent to LLM

Full text of each chunk that will be included in the prompt.

In [84]:
for i, h in enumerate(final_hits):
    words = len(h["text"].split())
    print(f"{'='*80}")
    print(f"[Doc{i+1}]  {h['chunk_id']}")
    print(f"URL:    {h['url']}")
    print(f"Words:  {words}")
    print(f"{'='*80}")
    print(h["text"][:2000])
    if len(h["text"]) > 2000:
        print(f"\n... ({len(h['text']) - 2000} chars truncated)")
    print()

[Doc1]  chunk_766_FAQs_|_DSI
URL:    https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
Words:  458
Online Program
If I am a student in the In-Person Master's in Applied Data Science Program, may I take courses in the Online Program? Conversely, if I am a student in the Online Program, may I take courses in the In-Person program?
Currently, students may only take Master’s in Applied Data Science courses in the modality in which they are officially enrolled.
Do I need to be a US citizen or permanent resident to apply to Master's in Applied Data Science Online Program?
No, students do not have to be US citizen or resident to partake in the
Online Program
. Please note that the
Online Program
is
not
eligible for visa sponsorship.
How will enrolling in Master's in Applied Data Science Online Program impact my schedule? Are classes held synchronously, asynchronously, or both?
Classes generally take place on evenings and weekends in order to allow ou

## Step 9: LLM Generation + Citation Parsing

Build prompt, call Gemini, parse citations. Requires `GOOGLE_API_KEY` in `.env`.

In [85]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from prompt import build_prompt, parse_citations

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_output_tokens=512,
    google_api_key=os.getenv("GOOGLE_API_KEY", ""),
)

# Build prompt
prompt = build_prompt(QUERY, final_hits)
print("=" * 80)
print("PROMPT (first 2000 chars):")
print("=" * 80)
print(prompt[:2000])
if len(prompt) > 2000:
    print(f"\n... ({len(prompt) - 2000} chars truncated)")
print(f"\nTotal prompt length: {len(prompt)} chars, ~{len(prompt.split())} words")

PROMPT (first 2000 chars):
You are a helpful assistant for University of Chicago's MS in Applied Data Science program.
Use the provided context below to answer the user question. Answer concisely and directly. IMPORTANT: Do NOT start with disclaimers or statements about what the context lacks. If the context contains relevant information -- even under different terminology -- synthesize all relevant details. If the context only partially addresses the question, answer the part you can and clearly state which part is not covered. Answer in the same language as the user's question. Keep proper nouns, program names, course titles, and academic terms in their original English form. Format your answer in Markdown.
At the very end of your answer, on a new line, list ONLY the document labels you actually used (e.g. [Doc1][Doc3]). Do not include documents you did not reference. If you cannot answer the question from the context, do NOT cite any documents.

[CONTEXT]
[Doc1] Online Program
If I 

In [86]:
# Call LLM (use await directly in Jupyter's async event loop)
raw_answer = (await llm.ainvoke(prompt)).content

print("=" * 80)
print("RAW LLM OUTPUT:")
print("=" * 80)
print(raw_answer)

RAW LLM OUTPUT:
The Master's in Applied Data Science program prepares students to work at the intersection of data science, machine learning, and artificial intelligence. The program's curriculum is designed to provide a balance between theory and rigorous application experience, with coursework and a Capstone project. Employers have high expectations for technical competence, communication and influence skills, and exposure to advanced Data Science evolving technologies.

The program offers various electives that evolve with the data science landscape, including:
*   Advanced Computer Vision with Deep Learning
*   Advanced Machine Learning and Artificial Intelligence
*   Bayesian Machine Learning with Generative AI Applications
*   Data Science for Algorithmic Marketing
*   Data Visualization Techniques
*   Digital Marketing Analytics in Theory and Practice
*   Quantitative Finance: Methods and Applications
*   Data Science for Healthcare
*   Machine Learning Operations
*   Next-Gen N

In [87]:
# Parse citations
clean_answer, sources, references = parse_citations(raw_answer, final_hits)

print("=" * 80)
print("CLEAN ANSWER:")
print("=" * 80)
print(clean_answer)

print(f"\n{'='*80}")
print("CITED SOURCES:")
print("=" * 80)
if sources:
    for i, s in enumerate(sources):
        print(f"  {i+1}. {s}")
else:
    print("  (no sources cited)")

print(f"\nREFERENCES:")
for ref in references:
    print(f"  - {ref['title'][:60]}  →  {ref['url']}")

CLEAN ANSWER:
The Master's in Applied Data Science program prepares students to work at the intersection of data science, machine learning, and artificial intelligence. The program's curriculum is designed to provide a balance between theory and rigorous application experience, with coursework and a Capstone project. Employers have high expectations for technical competence, communication and influence skills, and exposure to advanced Data Science evolving technologies.

The program offers various electives that evolve with the data science landscape, including:
*   Advanced Computer Vision with Deep Learning
*   Advanced Machine Learning and Artificial Intelligence
*   Bayesian Machine Learning with Generative AI Applications
*   Data Science for Algorithmic Marketing
*   Data Visualization Techniques
*   Digital Marketing Analytics in Theory and Practice
*   Quantitative Finance: Methods and Applications
*   Data Science for Healthcare
*   Machine Learning Operations
*   Next-Gen NLP

## Utility: Look up any chunk by ID

In [90]:
# ✏️ Change this to look up any chunk
LOOKUP_ID = "chunk_787_FAQs_|_DSI"

for c in chunk_records:
    if c["chunk_id"] == LOOKUP_ID:
        meta = c["metadata"]
        print(f"Chunk ID:    {c['chunk_id']}")
        print(f"Type:        {meta.get('chunk_type', '')}")
        print(f"Page Title:  {meta.get('page_title', '')}")
        print(f"Heading:     {meta.get('heading', '')}")
        print(f"Labels:      {meta.get('labels', [])}")
        print(f"URL:         {_get_url(meta)}")
        print(f"Words:       {len(c['text'].split())}")
        print(f"\n{'='*80}")
        print(c["text"])
        break
else:
    print(f"Chunk '{LOOKUP_ID}' not found.")

Chunk ID:    chunk_787_FAQs_|_DSI
Type:        section
Page Title:  FAQs | DSI
Heading:     FAQs
Labels:      ['other']
URL:         https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
Words:       26

[FAQs]
FAQs
Master’s in Applied Data Science FAQs
Learn more about what makes our program unique.
David Uminsky, PhD – UChicago Data Science Institute, Executive Director


## Utility: Search chunks by keyword

In [89]:
# ✏️ Change this to search for any keyword in chunk text
KEYWORD = "Real Time Intelligent Systems"

matches = []
for c in chunk_records:
    if KEYWORD.lower() in c["text"].lower():
        matches.append(c)

print(f"Found {len(matches)} chunks containing '{KEYWORD}':\n")
for c in matches[:20]:
    meta = c["metadata"]
    ctype = meta.get("chunk_type", "")
    # Find the keyword in context
    idx = c["text"].lower().index(KEYWORD.lower())
    start = max(0, idx - 60)
    end = min(len(c["text"]), idx + len(KEYWORD) + 60)
    context = c["text"][start:end].replace("\n", " ")
    print(f"  {c['chunk_id'][:55]}")
    print(f"    type={ctype}  ...{context}...")
    print()

Found 18 chunks containing 'Real Time Intelligent Systems':

  chunk_460_In-Person_Program_|_DSI
    type=accordion  ...rojects using R. Basic familiarity with R is a requirement. Real Time Intelligent Systems Developing end-to-end automation and intelligent systems is...

  chunk_476_In-Person_Program_|_DSI
    type=accordion_course  ...Sample Elective Courses — Real Time Intelligent Systems Developing end-to-end automation and intelligent systems is...

  chunk_503_Online_Program_|_DSI
    type=accordion  ...e Classification and Watson Natural Language Understanding. Real Time Intelligent Systems Developing end-to-end automation and intelligent systems is...

  chunk_531_Online_Program_|_DSI
    type=section  ...g Operations, Next-Gen NLP: LLM and Agentic AI in Practice, Real Time Intelligent Systems, Deep Reinforcement Learning, Supply Chain Optimization. Ca...

  chunk_630_Course_Progressions_|_DSI
    type=accordion  ...tions, Natural Language Processing and Cognitive Computing, R